# Huggingface Movie Project

This project is to learn more about qdrant, pandas, and huggingface

## Setup

In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "NiklasAbraham/MoviePlotEmbeddingsDataset"

path = hf_hub_download(repo_id=repo_id, filename="final_dataset.csv", repo_type="dataset")
df = pd.read_csv(path)

print(df.shape)
print(df.columns.tolist())
df[["title", "year", "genre"]].head()

In [ ]:
df = df.dropna(subset=["plot", "year", "genre"])
df = df[df["plot"].str.len() > 200]

random_sample_seed = 42
sample = df.sample(n = 4000, random_state = random_sample_seed).reset_index(drop=True)
sample.shape

In [ ]:
df["decade"] = (df["year"] // 10) * 10
sample = (
  df.groupby("decade", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), 400), random_state=random_sample_seed))
    .reset_index(drop=True)
)
sample.shape

In [ ]:
cols=["movie_id", "title", "year", "genre", "plot", "directors"]
sample = sample[cols].reset_index(drop = True)
sample.to_parquet("data/movies_sample.parquet")

# Data Cleaning

In [ ]:
sample = pd.read_parquet("data/movies_sample.parquet")
sample["genre"].head(10).tolist()

In [ ]:
sample["genre"] = sample["genre"].str.split(",")
sample["genre"] = sample["genre"].apply(lambda x : [item.lower().strip() for item in x])
sample["genre"].head(10).tolist()

# EMBEDDING SETUP

In [ ]:
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

vec_path = Path("data/vectors.npy")

if vec_path.exists():
  vectors = np.load(vec_path)
else:
  model = SentenceTransformer("intfloat/multilingual-e5-small")
  texts = ["passage: " + p for p in sample["plot"]]
  vectors = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
  )
  np.save(vec_path, vectors)

vectors.shape

In [ ]:
# Quick check if vectors are normalized -- if yes, should yield ~1
np.linalg.norm(vectors, axis=1)[:5]

In [ ]:
sample["genre"].head(10).tolist()

# QDrant Setup

In [ ]:

from qdrant_client.models import Distance, PayloadSchemaType, VectorParams
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

if not client.collection_exists(collection_name="movies"):
  client.create_collection(
    collection_name="movies",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
  )

client.get_collections()

In [ ]:
client.create_payload_index(
  collection_name="movies",
  field_schema=PayloadSchemaType.INTEGER,
  field_name="year",
)

client.create_payload_index(
  collection_name="movies",
  field_schema=PayloadSchemaType.KEYWORD,
  field_name="genres",
)

In [ ]:
# Remove dirty genres
def clean_genres(g):
  out = []
  for x in list(g):
    x = str(g).strip().lower().removesuffix(" film")
    if x and x not in out: # do not add duplicates
      out.append(x)
  return out

sample["genres"] = sample["genre"].apply(clean_genres)
sample = sample[sample["genres"].str.len() > 0]
sample.shape


In [ ]:
mask = sample["genres"].str.len() > 0
vectors = vectors[mask.to_numpy()]
sample = sample[mask].reset_index(drop= True)
assert len(sample) == len(vectors)

In [ ]:
from qdrant_client.models import PointStruct


points = [
  PointStruct(
    id=i, 
    vector=vectors[i].tolist(), 
    payload={
      "title": row.title, 
      "year" : int(row.year), 
      "genres": list(row.genres),
      }
    )
  for i, row in enumerate(sample.itertuples(index=False)) 
]

points[:5]

In [ ]:
client.upload_points(collection_name="movies", points=points, batch_size=256)
client.count(collection_name="movies")

In [ ]:
# Quick check
info = client.get_collection(collection_name="movies")

print(info.payload_schema)

In [ ]:
print(client.retrieve(collection_name="movies", ids=[0], with_payload=True))
print(sample.iloc[0][["title", "year"]])

# First Searches

In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")

def search(text: str, limit = 5, **kwargs):
  q = model.encode("query: " + text, normalize_embeddings=True).tolist()
  res = client.query_points("movies", query=q, limit=limit, with_payload=True, **kwargs)
  return [(p.payload["title"], p.payload["year"], round(p.score, ndigits=2)) for p in res.points]

search("a robot learns how to love while cleaning an abandoned earth")

In [ ]:
search("The plot deals with issues such as the pressures of social media, the lure of superficial beauty and the importance of being yourself")

In [ ]:
search("Franz and coppelia love story")

In [ ]:
res = sample[sample["title"] == "Coppelia"]["plot"]
print(res.iloc[0].split(" ").__len__())

In [ ]:
model.max_seq_length

In [ ]:
plots = []

for i, row in enumerate(sample.itertuples(index=False)):
  if i > 50:
    break
  plot = ". ".join(row.plot.split(". ")[:2])
  plots.append((row.title, plot))

def get_ranking(title, ranking):
  return 

not_matched = 0

for real_title, plot in plots:
  res = search(text=plot)
  is_match = False
  title_concat = ""
  for title, year, score in res:
    title_concat += title + ", "
    if(title == real_title):
      is_match = True

  title_concat = title_concat[:-2]

  if is_match:
    print(f"Matched! {real_title} found in searches. All the titles: [{title_concat}]")
  else:
    not_matched += 1
    print(f"Not matched!")

print(f"Not matched number: {not_matched}")


In [ ]:
import random, numpy as np

def search_ids(text, limit = 10):
  q = model.encode("query: " + text, normalize_embeddings=True).tolist()
  return [p.id for p in client.query_points("movies", query = q, limit = limit).points]

rng = random.Random()
test_ids = rng.sample(range(len(sample)), 50)

scores = []
for i in test_ids:
  q_text = ". ".join(sample.iloc[i]["plot"].split(". ")[:2])
  found = search_ids(q_text)
  scores.append(1/ (found.index(i) + 1) if i in found else 0)

print("MRR@10", np.mean(scores))

In [ ]:
def mrr_for(make_query, n = 50, seed = 0):
  rng =  random.Random(seed)
  ids = [i for i in range(len(sample)) if len(sample.iloc[i]["plot"].split(". ")) >= 6]
  scores = []
  for i in rng.sample(ids, n):
    sents = sample.iloc[i]["plot"].split(". ")
    found = search_ids(make_query(sents))
    scores.append(1/(found.index(i) + 1) if i in found else 0)
  return np.mean(scores)

print("first 2 sentences:", mrr_for(lambda s: ". ".join(s[:2])))
print("first 1 sentence: ", mrr_for(lambda s: s[0]))
print("middle sentence:  ", mrr_for(lambda s: s[len(s)//2]))
print("last sentence:    ", mrr_for(lambda s: s[-1]))

In [ ]:
tok_len = np.array([len(model.tokenizer(p)["input_ids"]) for p in sample["plot"]])
print(model.max_seq_length, np.percentile(tok_len, [50, 90, 99]), (tok_len > model.max_seq_length).mean())


In [ ]:
def mrr_for(make_query, pool, n=50, seed=0):
  rng = random.Random(seed)
  scores = []
  for i in rng.sample(pool, min(n,len(pool))):
    sents = sample.iloc[i]["plot"].split(". ")
    found = search_ids(make_query(sents))
    scores.append(1/ (found.index(i) + 1) if i in found else 0)
  return np.mean(scores)

ok = [i for i in range(len(sample)) if len(sample.iloc[i]["plot"].split(". ")) >= 6] # 6 sentences minimum
short = [i for i in ok if tok_len[i] < 400]
long = [i for i in ok if tok_len[i] > 600]

print("last sentence short plots: ", mrr_for(lambda s: s[-1], short))
print("last sentence long plots: ", mrr_for(lambda s: s[-1], long))


In [ ]:
print("last sentence, super short", mrr_for(lambda s: s[-1], [i for i in ok if tok_len[i] < 300]))

In [ ]:
short_ids = [i for i in ok if tok_len[i] < 300][:200]

def rank_of(i, q_text, doc_text):
  q = model.encode("query: " + q_text, normalize_embeddings=True)
  v = vectors.copy()
  v[i] = model.encode("passage: " + doc_text, normalize_embeddings=True)
  sims = v @ q
  return int((sims > sims[i]).sum()) + 1

def rr(rank) : return 1 / rank if rank <= 10 else 0

orig, moved, first_orig = [], [], []
helped = 0
hurt = 0
for i in short_ids:
    s = sample.iloc[i]["plot"].split(". ")
    first = s[0].rstrip(".")
    last = s[-1].rstrip(".")
    rest = s[:-1]
    first_orig.append(rr(rank_of(i, first, sample.iloc[i]["plot"])))

    orig_num = rr(rank_of(i, last, ". ".join(rest + [last])))
    moved_num = rr(rank_of(i, last, ". ".join([last] + rest)))
    orig.append(orig_num)
    moved.append(moved_num)

    is_helped = moved_num > orig_num
    if is_helped:
       helped += 1
    elif orig_num < moved_num:
       hurt += 1

print("first sentence:   ", np.mean(first_orig))
print("original order:   ", np.mean(orig))
print("last moved front: ", np.mean(moved))
print("helped: ", helped)
print("hurt: ", hurt)